# 02 — Model Training & Evaluation
**Project:** Farmer Loan Default Predictor  
**Author:** Rajanithi N · AI & Data Science · DSU-SET

This notebook covers data preprocessing, feature engineering, model selection, training, evaluation, and saving the final model pipeline.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

sns.set_style('whitegrid')
df = pd.read_csv('../data/loan_data.csv')
print('Dataset loaded:', df.shape)
df.head(3)

## 1. Data Preprocessing

In [ ]:
# --- Step 3a: Handle Missing Values ---
print('Missing values before cleaning:')
print(df.isnull().sum())

# Fill numeric columns with their median value
df.fillna(df.median(numeric_only=True), inplace=True)

# Drop any remaining rows that still have nulls (e.g. in categorical columns)
df.dropna(inplace=True)

print('\nMissing values after cleaning:', df.isnull().sum().sum())
print('Dataset shape after cleaning:', df.shape)

In [ ]:
# --- Step 3b: Encode Categorical Features using LabelEncoder ---
df_model = df.copy()

cat_cols = ['crop_type', 'repayment_history', 'soil_quality', 'irrigation_type', 'state']

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])
    print(f'{col}: {list(le.classes_)}')

print('\nEncoding complete.')
df_model.head(3)

## 2. Feature Engineering

In [ ]:
# Derived features
df_model['loan_to_income_ratio'] = df_model['loan_amount'] / df_model['annual_income']
df_model['income_per_acre']      = df_model['annual_income'] / df_model['land_area_acres']
df_model['loan_per_acre']        = df_model['loan_amount'] / df_model['land_area_acres']

FEATURES = [
    'age', 'land_area_acres', 'annual_income', 'crop_type',
    'loan_amount', 'loan_tenure_months', 'previous_loans',
    'repayment_history', 'soil_quality', 'irrigation_type',
    'credit_score', 'state', 'rainfall_mm', 'avg_temp_celsius',
    'loan_to_income_ratio', 'income_per_acre', 'loan_per_acre'
]

X = df_model[FEATURES]
y = df_model['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Model Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
for name, clf in models.items():
    cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='roc_auc')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    results.append({
        'Model':         name,
        'CV AUC (mean)': round(cv_scores.mean(), 4),
        'CV AUC (std)':  round(cv_scores.std(), 4),
        'Test Accuracy': round(accuracy_score(y_test, y_pred), 4)
    })

results_df = pd.DataFrame(results).sort_values('CV AUC (mean)', ascending=False)
print(results_df.to_string(index=False))

## 4. Train Final Model (Random Forest)

In [ ]:
best_model = RandomForestClassifier(
    n_estimators=200, max_depth=8,
    min_samples_split=4, class_weight='balanced', random_state=42
)
best_model.fit(X_train, y_train)

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('Accuracy :', accuracy_score(y_test, y_pred))
print('ROC AUC  :', roc_auc_score(y_test, y_proba))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

## 5. Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'ROC AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > importances.quantile(0.75) else '#3498db' for v in importances]
importances.plot(kind='barh', color=colors, edgecolor='black', alpha=0.85)
plt.title('Feature Importances — Random Forest', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
joblib.dump(best_model, '../loan_model.pkl')
print('✅ Model saved to loan_model.pkl')